## About Guided Cursor

Welcome to **Guided Cursor**, an AI-powered platform designed to guide you through programming problems step by step. As with all AI systems, responses may not always be perfectly accurate. You are encouraged to think critically and verify all output independently.

**Privacy.** We may collect anonymised usage data to improve the platform and support academic research into AI-assisted pedagogy. No data will be shared outside the project team. Your usage and performance will **not** be disclosed to module leaders and will have **no bearing** on your academic grades.

**Data Retention.** This platform may be taken offline at the end of the academic term, and all stored data may be permanently deleted. Please back up any materials you wish to keep in advance.

**Contact.** For any questions or concerns, please reach out to **hello@guidedcursor.studio**.

# The Discrete Fourier Transform and Fast Fourier Transform

**Learning objectives**

By the end of this notebook you will be able to:

- Explain the motivation for frequency analysis: decomposing a signal into its sinusoidal components
- Describe the core intuition behind the DFT: correlation with test sinusoids
- Implement the naive DFT using nested loops and verify it recovers known frequencies
- Explain why the naive DFT is computationally expensive and verify its $\mathcal{O}(N^2)$ scaling
- Understand the FFT as an $\mathcal{O}(N \log N)$ algorithm that computes the same result as the DFT
- Use NumPy's FFT functions to analyse real-world signals in practice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from fourier_hints import show_hint
from fourier_verify import check_naive_dft, check_noisy_spectrum

## 1. Motivation: From Signals to Frequencies

Given a signal (a sequence of measurements over time), how do we determine which frequencies are present in it? This is the fundamental question of **frequency analysis**, and it arises everywhere: in audio processing, telecommunications, medical imaging, and physics.

We begin with a concrete example. Suppose someone hands you a recording that sounds like a complex tone. It turns out the signal is the sum of several pure sine waves, each with a different frequency and amplitude. Your task: figure out which sine waves were mixed together, and how strong each one is.

In [ ]:
# Complete code -- just run this cell
fs = 500                       # sampling rate (Hz)
t = np.arange(0, 1, 1 / fs)   # 1 second, 500 samples
N = len(t)

# Three sine-wave components
s1 = 1.0 * np.sin(2 * np.pi * 5 * t)    # 5 Hz, amplitude 1.0
s2 = 0.6 * np.sin(2 * np.pi * 12 * t)   # 12 Hz, amplitude 0.6
s3 = 0.3 * np.sin(2 * np.pi * 20 * t)   # 20 Hz, amplitude 0.3
signal = s1 + s2 + s3

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: individual components
axes[0].plot(t, s1, color="steelblue", linewidth=1.5, label="5 Hz (A=1.0)")
axes[0].plot(t, s2, color="crimson", linewidth=1.5, label="12 Hz (A=0.6)")
axes[0].plot(t, s3, color="forestgreen", linewidth=1.5, label="20 Hz (A=0.3)")
axes[0].set_xlabel("Time (s)", fontsize=13)
axes[0].set_ylabel("Amplitude", fontsize=13)
axes[0].set_title("Individual components", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].tick_params(labelsize=11)
axes[0].grid(True, alpha=0.3)

# Right: composite signal
axes[1].plot(t, signal, color="steelblue", linewidth=1.5)
axes[1].set_xlabel("Time (s)", fontsize=13)
axes[1].set_ylabel("Amplitude", fontsize=13)
axes[1].set_title("Composite signal: can you identify the components?", fontsize=14)
axes[1].tick_params(labelsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The composite signal on the right is all we observe. How do we recover the individual frequencies and amplitudes? This is exactly what the Discrete Fourier Transform tells us.

## 2. Sine Wave Basics

Before diving into the DFT, we need a shared vocabulary for describing sine waves.

A sine wave is defined by three parameters:

| Parameter | Symbol | What it controls |
|-----------|--------|------------------|
| **Amplitude** | $A$ | Height of the wave (how loud / how strong) |
| **Frequency** | $f$ | Number of complete cycles per second (Hz) |
| **Phase** | $\phi$ | Horizontal shift (where the wave starts) |

$$x(t) = A \sin(2\pi f t + \phi)$$

The plots below show the effect of varying each parameter independently.

In [ ]:
# Complete code -- just run this cell
t_demo = np.linspace(0, 1, 500)
colours = ["steelblue", "crimson", "forestgreen"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Varying amplitude
for A, col in zip([0.5, 1.0, 2.0], colours):
    axes[0].plot(t_demo, A * np.sin(2 * np.pi * 2 * t_demo), color=col,
                 linewidth=1.5, label=f"A={A}")
axes[0].set_title("Varying amplitude ($f=2$ Hz, $\\phi=0$)", fontsize=13)
axes[0].set_xlabel("Time (s)", fontsize=12)
axes[0].set_ylabel("$x(t)$", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Varying frequency
for f_val, col in zip([1, 3, 5], colours):
    axes[1].plot(t_demo, np.sin(2 * np.pi * f_val * t_demo), color=col,
                 linewidth=1.5, label=f"f={f_val} Hz")
axes[1].set_title("Varying frequency ($A=1$, $\\phi=0$)", fontsize=13)
axes[1].set_xlabel("Time (s)", fontsize=12)
axes[1].set_ylabel("$x(t)$", fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Varying phase
phase_labels = ["$0$", "$\\pi/4$", "$\\pi/2$"]
for phi, col, lbl in zip([0, np.pi/4, np.pi/2], colours, phase_labels):
    axes[2].plot(t_demo, np.sin(2 * np.pi * 2 * t_demo + phi), color=col,
                 linewidth=1.5, label=f"$\\phi$={lbl}")
axes[2].set_title("Varying phase ($A=1$, $f=2$ Hz)", fontsize=13)
axes[2].set_xlabel("Time (s)", fontsize=12)
axes[2].set_ylabel("$x(t)$", fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. The Core Idea: Correlation with Sinusoids

### 3.1 Intuition

Suppose we want to know whether a particular frequency $f_{\text{test}}$ is present in a signal. The idea is simple:

1. **Multiply** the signal point-by-point with a test sinusoid at frequency $f_{\text{test}}$.
2. **Sum** all the products.

If the test frequency matches a component of the signal, the products are consistently positive (or consistently negative), and the sum is large. If the test frequency is absent, the products oscillate between positive and negative values and cancel to near zero.

This is the same idea as the **dot product** measuring alignment between two vectors: we are measuring the "alignment" between the signal and a test sinusoid.

In [ ]:
# Complete code -- just run this cell
N_demo = 8
n_demo = np.arange(N_demo)
n_fine = np.linspace(0, N_demo - 1, 200)
x_demo = np.cos(2 * np.pi * 2 * n_demo / N_demo)  # frequency bin k=2

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Continuous wave shapes for visual context
wave_signal = np.cos(2 * np.pi * 2 * n_fine / N_demo)

# --- Matching frequency: k=2 ---
test_match = np.cos(2 * np.pi * 2 * n_demo / N_demo)
product_match = x_demo * test_match
wave_test_match = np.cos(2 * np.pi * 2 * n_fine / N_demo)

axes[0, 0].plot(n_fine, wave_signal, color="steelblue", alpha=0.3, linewidth=1)
axes[0, 0].plot(n_fine, wave_test_match, color="crimson", alpha=0.3, linewidth=1)
axes[0, 0].stem(n_demo, x_demo, linefmt="steelblue", markerfmt="o",
                basefmt="grey", label="Signal")
axes[0, 0].stem(n_demo, test_match, linefmt="crimson", markerfmt="^",
                basefmt="grey", label="Test (k=2)")
axes[0, 0].set_title("Signal and test sinusoid (k=2, matching)", fontsize=13)
axes[0, 0].legend(fontsize=10)
axes[0, 0].set_xlabel("n", fontsize=12)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].stem(n_demo, product_match, linefmt="forestgreen", markerfmt="s",
                basefmt="grey")
axes[0, 1].set_title(f"Element-wise product (sum = {np.sum(product_match):.2f})",
                     fontsize=13)
axes[0, 1].set_xlabel("n", fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# --- Non-matching frequency: k=1 ---
test_miss = np.cos(2 * np.pi * 1 * n_demo / N_demo)
product_miss = x_demo * test_miss
wave_test_miss = np.cos(2 * np.pi * 1 * n_fine / N_demo)

axes[1, 0].plot(n_fine, wave_signal, color="steelblue", alpha=0.3, linewidth=1)
axes[1, 0].plot(n_fine, wave_test_miss, color="crimson", alpha=0.3, linewidth=1)
axes[1, 0].stem(n_demo, x_demo, linefmt="steelblue", markerfmt="o",
                basefmt="grey", label="Signal")
axes[1, 0].stem(n_demo, test_miss, linefmt="crimson", markerfmt="^",
                basefmt="grey", label="Test (k=1)")
axes[1, 0].set_title("Signal and test sinusoid (k=1, non-matching)", fontsize=13)
axes[1, 0].legend(fontsize=10)
axes[1, 0].set_xlabel("n", fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].stem(n_demo, product_miss, linefmt="forestgreen", markerfmt="s",
                basefmt="grey")
axes[1, 1].set_title(f"Element-wise product (sum = {np.sum(product_miss):.2f})",
                     fontsize=13)
axes[1, 1].set_xlabel("n", fontsize=12)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 From Cosine and Sine to Complex Exponentials

Testing with cosine alone tells us the "in-phase" component; testing with sine tells us the "quadrature" (90-degree shifted) component. Instead of tracking two separate correlations, we can combine them using **Euler's formula**:

$$e^{-j\theta} = \cos\theta - j\sin\theta$$

where $j = \sqrt{-1}$.

By multiplying the signal with a complex exponential $e^{-j 2\pi k n / N}$, a single multiplication captures both the cosine and sine correlations simultaneously. This is the key insight that leads directly to the DFT formula.

## 4. The DFT Formula

### 4.1 Definition

The **Discrete Fourier Transform** of an $N$-point signal $x[n]$ is:

$$X[k] = \sum_{n=0}^{N-1} x[n] \, e^{-j 2\pi k n / N}, \qquad k = 0, 1, \ldots, N-1$$

Each term in the sum is exactly the correlation operation from Section 3:

- $x[n]$ is the signal sample at time index $n$
- $e^{-j 2\pi k n / N}$ is the test sinusoid at frequency index $k$
- The sum accumulates the correlation across all $N$ samples

The output $X[k]$ is a complex number for each frequency bin $k$. Its magnitude $|X[k]|$ tells us the strength of that frequency, and its angle $\angle X[k]$ tells us the phase.

The DFT can be written as a matrix-vector multiplication $\mathbf{X} = W \mathbf{x}$, where $W_{kn} = e^{-j 2\pi kn / N}$. The heatmap below shows the real and imaginary parts of this matrix for $N = 8$.

In [ ]:
# Complete code -- just run this cell
N_mat = 8
k_idx = np.arange(N_mat)
n_idx = np.arange(N_mat)
W = np.exp(-2j * np.pi * np.outer(k_idx, n_idx) / N_mat)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(W.real, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Real part (cosines)", fontsize=14)
axes[0].set_xlabel("Sample index $n$", fontsize=13)
axes[0].set_ylabel("Frequency bin $k$", fontsize=13)
axes[0].set_xticks(k_idx)
axes[0].set_yticks(k_idx)
axes[0].tick_params(labelsize=11)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(W.imag, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("Imaginary part (sines)", fontsize=14)
axes[1].set_xlabel("Sample index $n$", fontsize=13)
axes[1].set_ylabel("Frequency bin $k$", fontsize=13)
axes[1].set_xticks(k_idx)
axes[1].set_yticks(k_idx)
axes[1].tick_params(labelsize=11)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.suptitle("DFT matrix ($N = 8$)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Reading the heatmap**

The matrix has $N = 8$ rows and $N = 8$ columns. Each column corresponds to one of the 8 sample points in the input signal. Each row is a frequency template at a specific frequency.

Look at the left heatmap (real part / cosines):

| Row ($k$) | Visual pattern | What it represents |
|-----------|---------------|-------------------|
| Row 0 | All deep red ($+1$) | DC: the average level of the signal, with no oscillation. |
| Row 1 | Red to blue and back | 1 complete oscillation across the 8 samples. |
| Row 2 | Red-blue-red-blue | 2 complete oscillations, twice as fast as Row 1. |
| Row 4 | Alternating red and blue (densest) | Highest frequency: the fastest possible alternation with 8 samples. |

**Why is the matrix symmetric?**

Row 1 and Row 7, Row 2 and Row 6 look very similar (mirrored). This is a property of the DFT: for real-valued signals, the high-frequency rows are mirrors of the low-frequency rows. In practice, we only need to look at the first half of the output (rows 0 to $N/2$).

**What does the DFT actually compute?**

To compute $X[k]$ (the DFT output at frequency $k$), we take the **dot product** of the input signal (all $N$ sample points) with row $k$ of this matrix. If the signal oscillates at the same rate as row $k$, the dot product is large. If not, the positive and negative terms cancel out and the result is near zero. This is exactly the correlation idea from Section 3, applied once for every frequency.

### 4.2 Implementation

Algorithm:

1. Determine the length $N$ of the input signal.
2. Allocate an output array `X` of length $N$ (complex).
3. **Outer loop** over frequency bins $k = 0, 1, \ldots, N-1$.
4. **Inner loop** over time samples $n = 0, 1, \ldots, N-1$: accumulate $x[n] \cdot e^{-j 2\pi k n / N}$ into `X[k]`.
5. Return `X`.

Fill in the nested loops below.

In [ ]:
def naive_dft(x):
    x = np.asarray(x, dtype=complex)
    N = len(x)
    X = np.zeros(N, dtype=complex)

    # ===== YOUR CODE BELOW =====



    # ===== YOUR CODE ABOVE =====
    return X

In [ ]:
show_hint("naive_dft")

In [ ]:
check_naive_dft(naive_dft)

### 4.3 Recovering the Frequencies

Let us apply `naive_dft` to the composite signal from Section 1 and see whether it can recover the three constituent frequencies.

In [ ]:
# Complete code -- just run this cell
X_signal = naive_dft(signal)
freqs = np.arange(N) * fs / N
magnitudes = np.abs(X_signal)

# Plot only the first half (positive frequencies)
half = N // 2

plt.figure(figsize=(9, 5.5))
plt.stem(freqs[:half], magnitudes[:half], linefmt="steelblue",
         markerfmt="o", basefmt="grey")

# Annotate the three peaks
for f_peak in [5, 12, 20]:
    idx = int(f_peak * N / fs)
    plt.annotate(f"{f_peak} Hz", (freqs[idx], magnitudes[idx]),
                 textcoords="offset points", xytext=(8, 8),
                 fontsize=11, color="black",
                 arrowprops=dict(arrowstyle="->", color="black"))

plt.xlabel("Frequency (Hz)", fontsize=13)
plt.ylabel("$|X[k]|$", fontsize=13)
plt.title("Magnitude spectrum of the composite signal", fontsize=14)
plt.xlim(-1, 50)
plt.tick_params(labelsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The DFT has recovered the three frequencies and their relative amplitudes from the composite signal. This is exactly the inverse problem we posed in Section 1.

## 5. Computational Cost of the Naive DFT

The naive DFT has two nested loops, each running from 0 to $N-1$, giving $N^2$ complex multiplications and additions. The time complexity is $\mathcal{O}(N^2)$.

For small $N$ this is fine, but for practical signal lengths (thousands or millions of samples), $N^2$ quickly becomes prohibitive. Let us measure this.

In [ ]:
# Complete code -- just run this cell
sizes_dft = [64, 128, 256, 512, 1024]
times_dft = []

print(f"{'N':>6s}  {'Time (s)':>10s}")
print("-" * 20)

for n_size in sizes_dft:
    x_time = np.random.randn(n_size)
    start = time.time()
    naive_dft(x_time)
    elapsed = time.time() - start
    times_dft.append(elapsed)
    print(f"{n_size:6d}  {elapsed:10.4f}")

# Log-log plot
plt.figure(figsize=(7, 4))
plt.loglog(sizes_dft, times_dft, "o-", color="crimson", label="Naive DFT")
# O(N^2) reference line
n_ref = np.array(sizes_dft, dtype=float)
scale = times_dft[0] / sizes_dft[0]**2
plt.loglog(n_ref, scale * n_ref**2, "--", color="grey", label="$\\mathcal{O}(N^2)$ reference")
plt.xlabel("$N$", fontsize=13)
plt.ylabel("Time (s)", fontsize=13)
plt.title("Naive DFT execution time", fontsize=14)
plt.legend(fontsize=10)
plt.tick_params(labelsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAt N={sizes_dft[-1]}, the naive DFT takes {times_dft[-1]:.2f} seconds.")
print("For a typical audio file with millions of samples, we need a faster algorithm.")

## 6. The Fast Fourier Transform and Practical Usage

### 6.1 From $\mathcal{O}(N^2)$ to $\mathcal{O}(N \log N)$

The timing results above show that the naive DFT is far too slow for real-world signals. Fortunately, there is a much faster way to compute exactly the same result.

In 1965, Cooley and Tukey published an algorithm called the **Fast Fourier Transform (FFT)**. The key insight is that the DFT formula contains repeated calculations that can be shared. By splitting the input into even-indexed and odd-indexed samples, computing a smaller DFT on each half, and combining the results, the total work drops dramatically. This splitting can be applied recursively (similar to how merge sort works), reducing the cost from $\mathcal{O}(N^2)$ to $\mathcal{O}(N \log N)$.

The FFT computes **exactly the same output** as the naive DFT, just much faster. For a signal with one million samples, this means roughly 50,000 times fewer operations.

A supplementary notebook with the full derivation and implementation of the Cooley-Tukey FFT algorithm will be provided this week. For now, we will focus on using NumPy's highly optimised FFT implementation for practical signal analysis.

### 6.2 Key NumPy FFT Functions

| Function | Purpose |
|----------|--------|
| `np.fft.fft(x)` | Compute the FFT of signal `x` |
| `np.fft.ifft(X)` | Inverse FFT: reconstruct signal from spectrum |
| `np.fft.fftfreq(N, d)` | Generate the frequency array (in Hz, given sample spacing `d`) |
| `np.fft.rfft(x)` | FFT for real-valued signals (returns only the first half) |
| `np.fft.rfftfreq(N, d)` | Frequency array for `rfft` |

In [ ]:
# Complete code -- just run this cell
# Demonstrate the key NumPy FFT functions
x_demo3 = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
N_demo3 = len(x_demo3)
fs_demo3 = 100  # Hz

# fft and ifft are inverses
X_demo3 = np.fft.fft(x_demo3)
x_recovered = np.fft.ifft(X_demo3)
print("fft and ifft are inverses:")
print(f"  np.allclose(ifft(fft(x)), x) = {np.allclose(x_recovered, x_demo3)}")

# fftfreq generates the frequency axis (includes negative frequencies)
freqs_full = np.fft.fftfreq(N_demo3, d=1/fs_demo3)
print(f"\nfftfreq (full): {freqs_full}")

# rfft returns only the positive-frequency half (more efficient for real signals)
X_rfft = np.fft.rfft(x_demo3)
freqs_rfft = np.fft.rfftfreq(N_demo3, d=1/fs_demo3)
print(f"\nrfft output length: {len(X_rfft)} (vs fft: {len(X_demo3)})")
print(f"rfftfreq: {freqs_rfft}")
print(f"\nrfft is the efficient choice for real-valued signals: it returns")
print(f"only the non-redundant bins (N//2 + 1 = {N_demo3 // 2 + 1} instead of N = {N_demo3}).")

### 6.3 Exercise: Extracting Frequencies from a Noisy Signal

In practice, signals are contaminated by noise. The FFT can still identify the underlying frequency components, because noise spreads its energy across all frequencies while the signal components concentrate their energy at specific bins.

The cell below creates a clean signal (three sinusoidal components) and a noisy version. Your task is to compute the frequency axis and magnitude spectra using `np.fft.rfft` and `np.fft.rfftfreq`.

In [ ]:
# Complete code -- just run this cell
fs_noisy = 1000
T_noisy = 1.0
N_noisy = int(fs_noisy * T_noisy)
t_noisy = np.arange(N_noisy) / fs_noisy

# Clean signal: three sinusoidal components
clean = (1.0 * np.sin(2 * np.pi * 50 * t_noisy)
       + 0.5 * np.sin(2 * np.pi * 120 * t_noisy)
       + 0.3 * np.sin(2 * np.pi * 300 * t_noisy))

# Add Gaussian noise
np.random.seed(0)
noisy = clean + 1.5 * np.random.randn(N_noisy)

In [ ]:
# Complete code -- just run this cell
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: clean signal (time domain)
axes[0].plot(t_noisy[:200], clean[:200], color="steelblue", linewidth=1)
axes[0].set_xlabel("Time (s)", fontsize=12)
axes[0].set_ylabel("Amplitude", fontsize=12)
axes[0].set_title("Clean signal (time domain)", fontsize=13)
axes[0].tick_params(labelsize=10)
axes[0].grid(True, alpha=0.3)

# Right: noisy signal (time domain)
axes[1].plot(t_noisy[:200], noisy[:200], color="crimson", linewidth=0.8)
axes[1].set_xlabel("Time (s)", fontsize=12)
axes[1].set_ylabel("Amplitude", fontsize=12)
axes[1].set_title("Noisy signal (time domain)", fontsize=13)
axes[1].tick_params(labelsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compute the frequency axis and magnitude spectra using rfft.

# ===== YOUR CODE BELOW =====
freqs_n = ...
X_clean = ...
X_noisy = ...
# ===== YOUR CODE ABOVE =====

In [ ]:
show_hint("noisy_spectrum")

In [ ]:
check_noisy_spectrum(freqs_n, X_clean, X_noisy, N_noisy, fs_noisy)

In [ ]:
# Complete code -- just run this cell
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: clean spectrum (frequency domain)
axes[0].plot(freqs_n, X_clean, color="steelblue", linewidth=1)
axes[0].set_xlabel("Frequency (Hz)", fontsize=12)
axes[0].set_ylabel("$|X[k]|$", fontsize=12)
axes[0].set_title("Clean signal (frequency domain)", fontsize=13)
axes[0].set_xlim(0, 400)
axes[0].tick_params(labelsize=10)
axes[0].grid(True, alpha=0.3)

# Right: noisy spectrum (frequency domain)
axes[1].plot(freqs_n, X_noisy, color="crimson", linewidth=0.8)
for fp in [50, 120, 300]:
    idx_p = np.argmin(np.abs(freqs_n - fp))
    axes[1].annotate(f"{fp} Hz", (freqs_n[idx_p], X_noisy[idx_p]),
                     textcoords="offset points", xytext=(8, 8),
                     fontsize=10, color="black",
                     arrowprops=dict(arrowstyle="->", color="black"))
axes[1].set_xlabel("Frequency (Hz)", fontsize=12)
axes[1].set_ylabel("$|X[k]|$", fontsize=12)
axes[1].set_title("Noisy signal (frequency domain)", fontsize=13)
axes[1].set_xlim(0, 400)
axes[1].tick_params(labelsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Despite heavy noise in the time domain, the frequency-domain peaks at 50 Hz, 120 Hz, and 300 Hz are clearly visible above the noise floor.

## Closing Remarks

Fourier published his theory of heat conduction in 1807, decomposing complex temperature distributions into sums of simple sine waves. The idea was so powerful and so controversial that leading mathematicians of the time, including Lagrange, initially rejected it.

The algorithm that made Fourier analysis practical, the Fast Fourier Transform, was not published until 1965, when Cooley and Tukey showed how to reduce the cost from $\mathcal{O}(N^2)$ to $\mathcal{O}(N \log N)$. This single improvement transformed signal processing overnight. Within years, the FFT was being used to analyse seismic data, design communications systems, and process medical images.

Today, the FFT is one of the most important algorithms in all of computing. It underpins everything from MP3 compression and 4G/5G telecommunications to MRI imaging and gravitational wave detection.

### Summary

| Concept | Key idea | Complexity |
|---------|----------|------------|
| Correlation with sinusoids | Multiply signal by test sinusoid and sum | |
| Naive DFT | Nested loops over all bins and all samples | $\mathcal{O}(N^2)$ |
| Cooley-Tukey FFT | Exploits symmetry to split into smaller sub-problems | $\mathcal{O}(N \log N)$ |
| NumPy FFT | Optimised C implementation; use `rfft` for real signals | $\mathcal{O}(N \log N)$ |

**Key takeaways:**

- The DFT measures the correlation between a signal and sinusoids at each frequency bin. If a frequency is present, the correlation is large; if not, it cancels to near zero.
- The naive DFT costs $\mathcal{O}(N^2)$. The Cooley-Tukey FFT computes the same result in $\mathcal{O}(N \log N)$, making frequency analysis practical for large signals.
- In practice, use `np.fft.rfft` for real-valued signals and `np.fft.rfftfreq` to build the frequency axis.

**Questions to think about:**

- What happens when a signal's frequency does not fall exactly on a DFT bin? (Hint: look up "spectral leakage".)
- How would you extend the DFT to analyse two-dimensional data, such as images?
- The DFT analyses the entire signal at once. How could you track how frequencies change over time?